# 第 3 章：Qt 应用程序

## 使用说明

本 notebook 按网页和 PDF 的顺序完整收录 12 个小节、68 个原书代码块。原书代码保存在 Markdown 中；可直接运行的教学补充放在独立代码单元格里。

覆盖范围：3.1、3.2、3.3、3.3.1、3.3.2、3.3.3、3.3.4、3.3.5、3.3.6、3.3.7、3.3.8、3.4。

## 3.1 三个应用程序类

原书代码块共 4 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
app1 = QCoreApplication()
app2 = QCoreApplication() # 此处会出错，应用程序中已存在一个QCoreApplication 实例
```

**原书代码块 2**

```python
#1.实例化应用程序对象
app = Q*Application()
#2. 初始化其他组件
#如窗口、自定义QWidget对象等
#3.调用exec方法，进入事件循环
app.exec()
```

**原书代码块 3**

```python
sys.exit(app.exec())
```

**原书代码块 4**

```python
try:
    sys.exit(app.exec())
except SystemExit as e:
    print(f"程序即将退出，退出码：{e.code}")
```

#### 可运行补充

下面的单元格用于验证本节概念，不替代上方原书代码。

In [ ]:
from PySide6.QtCore import QCoreApplication
from PySide6.QtGui import QGuiApplication
from PySide6.QtWidgets import QApplication

print("QGuiApplication 是 QCoreApplication 子类：",
      issubclass(QGuiApplication, QCoreApplication))
print("QApplication 是 QGuiApplication 子类：",
      issubclass(QApplication, QGuiApplication))

## 3.2 示例：控制台应用程序

原书代码块共 5 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
TextInput = QEvent.Type(QEvent.registerEventType(QEvent.Type.User + 3))
```

**原书代码块 2**

```python
class TextInputEvent(QEvent):
    def __init__(self, type: QEvent.Type, text: str):
        super().__init__(type)
        self._text = text
    def text(self) -> str:
        return self._text
```

**原书代码块 3**

```python
def work():
    while True:
        try:
            s = input() #读取控制台的输入内容
            #获取当前应用程序实例
            curApp = QCoreApplication.instance()
            #如果输入的是"Q"，就退出应用程序
            if s == 'Q':
                curApp.exit(0)
                break
            #准备事件对象
            evt = TextInputEvent(TextInput, s)
            #向应用程序发送事件
            curApp.sendEvent(curApp, evt)
        except Exception as ex:
            print(ex)
```

**原书代码块 4**

```python
class MyApplication(QCoreApplication):
    def __init__(self):
        super().__init__()
    def event(self, arg: QEvent) -> bool:
        if arg.type() == TextInput:
            #此时 arg 的实际类型为TextInputEvent
            #因此可以访问text方法
            print(f'你输入了:{arg.text()}')
            return True
        return super().event(arg)
```

**原书代码块 5**

```python
if __name__ == "__main__":
    #实例化应用程序类
    app = MyApplication()
    #启动新线程
    th = Thread(target=work)
    th.start()
    #进入事件循环
    try:
        sys.exit(app.exec())
    except SystemExit as e:
        print(f"程序即将退出，退出码：{e.code}")
```

## 3.3 命令行参数

原书代码块共 6 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
#include <QApplication>
#include <QWidget>
// argc：命令行参数的数量
// argv：命令行参数列表
int main(int argc, char **argv)
{
    // 实例化 QApplication 时传递命令行参数
    QApplication app(argc, argv);
    // 其他初始化工作
    return QApplication::exec();
```

**原书代码块 2**

```python
#获取命令行参数列表
args = sys.argv
#实例化应用程序对象并传递命令行参数
app = QCoreApplication(args)
```

**原书代码块 3**

```python
python test.py abc xyz # arg1=abc, arg2=xyz
python test.py xyz abc # arg1=xyz, arg2=abc
```

**原书代码块 4**

```python
python test.py -a --b
```

**原书代码块 5**

```python
python test.py -a=1 -b=2
python test.py --a 1 --b 2
```

**原书代码块 6**

```python
python test.py --url=/includes --q file1 file2
```

### 3.3.1 示例：分析位置参数

原书代码块共 8 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
def addPositionalArgument(name: str, description: str, syntax: str)
```

**原书代码块 2**

```python
args = sys.argv
```

**原书代码块 3**

```python
app = QCoreApplication(args)
```

**原书代码块 4**

```python
parser = QCommandLineParser()
#添加帮助信息的选项支持
parser.addHelpOption()
#定义位置参数
parser.addPositionalArgument("action","操作类型，可选的值有copy、move、rename","<action>")
parser.addPositionalArgument("input", "输入文件", "<input file>")
parser.addPositionalArgument("output", "输出文件", "<output file>")
```

**原书代码块 5**

```python
result = parser.parse(app.arguments())
```

**原书代码块 6**

```python
if result:
    posargs = parser.positionalArguments()
    #必须是三个参数
    if len(posargs) != 3:
        print("参数个数不正确\n")
        #打印帮助信息
        parser.showHelp()
    else:
        action = posargs[0]
        if action == "copy":
            print(f"从{posargs[1]}复制到{posargs[2]}")
        elif action == "move":
            print(f"从{posargs[1]}移动到{posargs[2]}")
        elif action == "rename":
            print(f"将{posargs[1]}重命名为{posargs[2]}")
        else:
            print("action参数为未知指令\n")
            #打印帮助信息
            parser.showHelp()
```

**原书代码块 7**

```python
python app.py
```

**原书代码块 8**

```python
python app.py move /data/dxv.ts /foo/dic.ts
```

#### 可运行补充

下面的单元格用于验证本节概念，不替代上方原书代码。

In [ ]:
from PySide6.QtCore import QCommandLineParser

positional_parser = QCommandLineParser()
positional_parser.addHelpOption()
positional_parser.addPositionalArgument(
    "action", "操作类型，可选的值有 copy、move、rename", "<action>"
)
positional_parser.addPositionalArgument("input", "输入文件", "<input file>")
positional_parser.addPositionalArgument("output", "输出文件", "<output file>")

position_args = ["app.py", "move", "/data/dxv.ts", "/foo/dic.ts"]
result = positional_parser.parse(position_args)
posargs = positional_parser.positionalArguments()

if result and len(posargs) == 3:
    action = posargs[0]
    if action == "copy":
        print(f"从{posargs[1]}复制到{posargs[2]}")
    elif action == "move":
        print(f"从{posargs[1]}移动到{posargs[2]}")
    elif action == "rename":
        print(f"将{posargs[1]}重命名为{posargs[2]}")

In [ ]:
# Notebook 运行补充：本单元格可独立运行，会打开一个单独的进程。
import subprocess
import sys
from pathlib import Path

chapter_folder = '03-Qt应用程序'
relative_path = Path('command_line_demo.py')
start = Path.cwd().resolve()
chapter_dir = None
for base in (start, *start.parents):
    candidates = [
        base if base.name == chapter_folder else base / chapter_folder,
        base / "章节演示" / chapter_folder,
        base / "PySide6的学习版Web" / "章节演示" / chapter_folder,
    ]
    chapter_dir = next(
        (candidate for candidate in candidates if (candidate / "lesson.ipynb").exists()),
        None,
    )
    if chapter_dir is not None:
        break
if chapter_dir is None:
    raise FileNotFoundError(f"找不到章节目录：{chapter_folder}")

script = chapter_dir / relative_path
process = subprocess.Popen(
    [sys.executable, str(script), *['move', 'input.txt', 'output.txt']],
    cwd=script.parent,
)
print('3.3.1 完整命令行示例', "已启动，进程号：", process.pid)

### 3.3.2 添加选项参数

原书代码块共 1 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
QCommandLineOption(name)
QCommandLineOption(name, description, valueName, defaultValue)
QCommandLineOption(names)
QCommandLineOption(names, description, valueName, defaultValue)
QCommandLineOption(other)
```

### 3.3.3 示例：分析选项参数

原书代码块共 6 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
args = sys.argv
```

**原书代码块 2**

```python
app = QCoreApplication(args)
```

**原书代码块 3**

```python
cmdParser = QCommandLineParser()
```

**原书代码块 4**

```python
#添加第一个选项参数
#-p 或--oper
opt1 = QCommandLineOption(['p','oper'],'要执行的操作','operation')
cmdParser.addOption(opt1)
#添加第二个选项参数
#-n 或--name
opt2 = QCommandLineOption(['n','name'],'被执行的程序名称',"app name")
cmdParser.addOption(opt2)
```

**原书代码块 5**

```python
if cmdParser.parse(app.arguments()):
    if cmdParser.isSet(opt1):
        argnames ="、".join(opt1.names())
        print(f'{argnames} = {cmdParser.value(opt1)}')
    if cmdParser.isSet(opt2):
        argnames ="、".join(opt2.names())
        print(f'{argnames} = {cmdParser.value(opt2)}')
else:
    #命令行参数分析失败，输出错误信息
        print(cmdParser.errorText())
```

**原书代码块 6**

```python
python app.py -p close -n Zipper
python app.py --oper close --name zipper
python app.py --oper=close --name=Zipper
python app.py -n Zipper --oper close
```

#### 可运行补充

下面的单元格用于验证本节概念，不替代上方原书代码。

In [ ]:
from PySide6.QtCore import QCommandLineParser
from PySide6.QtCore import QCommandLineOption

option_parser = QCommandLineParser()
opt1 = QCommandLineOption(["p", "oper"], "要执行的操作", "operation")
opt2 = QCommandLineOption(["n", "name"], "被执行的程序名称", "app name")
option_parser.addOption(opt1)
option_parser.addOption(opt2)

option_args = ["app.py", "--oper", "close", "--name", "Zipper"]
if option_parser.parse(option_args):
    if option_parser.isSet(opt1):
        print(f"{'、'.join(opt1.names())} = {option_parser.value(opt1)}")
    if option_parser.isSet(opt2):
        print(f"{'、'.join(opt2.names())} = {option_parser.value(opt2)}")
else:
    print(option_parser.errorText())

### 3.3.4 帮助信息和版本信息

本节在原网页/PDF 中没有独立代码块。

### 3.3.5 示例：显示帮助信息

原书代码块共 12 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
args = sys.argv
```

**原书代码块 2**

```python
app = QCoreApplication(args)
```

**原书代码块 3**

```python
#设置应用程序名称
QCoreApplication.setApplicationName('My App')
#设置版本号
QCoreApplication.setApplicationVersion("1.0.3")
```

**原书代码块 4**

```python
parser = QCommandLineParser()
```

**原书代码块 5**

```python
parser.setApplicationDescription("示例应用程序")
```

**原书代码块 6**

```python
helpopt = parser.addHelpOption()
vsopt = parser.addVersionOption()
```

**原书代码块 7**

```python
parser.addPositionalArgument('target','操作目标')
```

**原书代码块 8**

```python
widopt = QCommandLineOption(['w','width'], '宽度', "target's width", '15cm')
parser.addOption(widopt)
radopt = QCommandLineOption(['r','radius'],'半径', "target's radius", '0cm')
parser.addOption(radopt)
```

**原书代码块 9**

```python
if parser.parse(app.arguments()):
    #是否打印帮助信息
    if parser.isSet(helpopt):
        parser.showHelp()
    #是否打印版本信息
    if parser.isSet(vsopt):
        parser.showVersion()
    #是否存提供了width 参数
    if parser.isSet(widopt):
        val = parser.value(widopt)
        print(f'width = {val}')
    #是否提供了 radius 参数
    if parser.isSet(radopt):
        val = parser.value(radopt)
        print(f'radius = {val}')
    #打印位置参数
    posargs = parser.positionalArguments()
    if len(posargs) > 0:
        print(f'位置参数：{" ".join(posargs)}')
else:
    #打印错误信息
    print(parser.errorText())
    #打印帮助信息
    parser.showHelp()
```

**原书代码块 10**

```python
python demo.py --help
```

**原书代码块 11**

```python
python demo.py -v
```

**原书代码块 12**

```python
python demo.py Compute --width=100cm -r 50cm
```

### 3.3.6 parse 方法与 process 方法

原书代码块共 1 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
args = sys.argv
app = QCoreApplication(args)
app.setApplicationVersion("2.0.0")
app.setApplicationName("DemoApp")
cmdParser = QCommandLineParser()
cmdParser.addHelpOption()
cmdParser.addVersionOption()
cmdParser.process(app)
```

### 3.3.7 示例：通过命令行参数运行其他应用程序

原书代码块共 12 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
test.py abc --x -data 1234.
```

**原书代码块 2**

```python
args = sys.argv
```

**原书代码块 3**

```python
app = QCoreApplication(args)
```

**原书代码块 4**

```python
app.setApplicationName("ExeCmd")
app.setApplicationVersion("2.0.0")
```

**原书代码块 5**

```python
cmdParser = QCommandLineParser()
```

**原书代码块 6**

```python
cmdParser.setOptionsAfterPositionalArgumentsMode(
    QCommandLineParser.OptionsAfterPositionalArgumentsMode.ParseAsPositionalArguments
)
```

**原书代码块 7**

```python
cmdParser.addHelpOption()
cmdParser.addVersionOption()
```

**原书代码块 8**

```python
cmdParser.addPositionalArgument(
    'command', '要执行的命令', '<command> [args...]'
)
```

**原书代码块 9**

```python
cmdParser.process(app)
```

**原书代码块 10**

```python
posargs = cmdParser.positionalArguments()
```

**原书代码块 11**

```python
if len(posargs) > 0:
    process = QProcess()
    #第一个参数是要执行的程序
    process.setProgram(posargs[0])
    #查看有没有要传递的参数
    if len(posargs) > 1:
        process.setArguments(posargs[1:])
    #启动进程
    process.start()
    isstarted = process.waitForStarted()
    if isstarted:
        #等待执行完成
        result = process.waitForFinished()
        #打印执行结果
        if result:
            arr = process.readAll()
            #解码出字符串
            # Windows 上默认使用 GBK 编码
            #Linux 上请使用 UTF-8 编码
            text = arr.data().decode('gbk')
            print(text)
    else:
        print(process.error())
```

**原书代码块 12**

```python
Demo.py cmd.exe /c ping www.qq.com
```

### 3.3.8 示例：根据命令行参数设定窗口的呈现方式

原书代码块共 11 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
args = sys.argv
```

**原书代码块 2**

```python
cmdParser = QCommandLineParser()
```

**原书代码块 3**

```python
optNormal = QCommandLineOption(['n','normal'],'显示常规窗口')
cmdParser.addOption(optNormal)
```

**原书代码块 4**

```python
optMinim = QCommandLineOption(['m','minimize'],'最小化窗口')
cmdParser.addOption(optMinim)
```

**原书代码块 5**

```python
optMaxim = QCommandLineOption(['x','maximize'], '最大化窗口')
cmdParser.addOption(optMaxim)
```

**原书代码块 6**

```python
result = cmdParser.parse(args)
```

**原书代码块 7**

```python
#创建QGuiApplication 实例
app = QGuiApplication()
#创建窗口实例
window = QWindow()
#设置窗口大小
window.resize(452, 365)
#设置窗口标题
window.setTitle("示例应用程序")
```

**原书代码块 8**

```python
if result: #命令行参数解析成功
    #根据命令行参数决定窗口状态
    if cmdParser.isSet(optNormal):
        window.showNormal()
    elif cmdParser.isSet(optMinim):
        window.showMinimized()
    elif cmdParser.isSet(optMaxim):
        window.showMaximized()
    else:  # 默认正常显示
        window.showNormal()
else:
    #如果命令行参数无效，将以默认方式显示窗口
    window.show()
```

**原书代码块 9**

```python
sys.exit(app.exec())
```

**原书代码块 10**

```python
python app.py --maximize
```

**原书代码块 11**

```python
python app.py -n
```

## 3.4 图形化应用程序

原书代码块共 2 个，以下按网页/PDF 顺序完整保留。其中有些是局部片段、声明或对照代码，不保证能够单独运行。

**原书代码块 1**

```python
from PySide6.QtWidgets import QApplication, QPushButton, QMessageBox, QWidget
#创建应用程序对象
thisApp = QApplication()
#创建主窗口
mainwin = QWidget()
#设置窗口大小和标题栏文本
mainwin.resize(150, 62)
mainwin.setWindowTitle('Test App')
#创建按钮组件实例
btn = QPushButton(mainwin)
btn.setText("请单击这里")
btn.move(12, 8)
#响应clicked信号
btn.clicked.connect(lambda: QMessageBox.information(
    mainwin,
    '提示信息',
    '你已单击按钮'
))
```

**原书代码块 2**

```python
#显示窗口
mainwin.showNormal()
#进入事件循环
thisApp.exec()
```

#### 可运行补充

下面的单元格用于验证本节概念，不替代上方原书代码。

In [ ]:
# Notebook 运行补充：本单元格可独立运行，会打开一个单独的进程。
import subprocess
import sys
from pathlib import Path

chapter_folder = '03-Qt应用程序'
relative_path = Path('demo.py')
start = Path.cwd().resolve()
chapter_dir = None
for base in (start, *start.parents):
    candidates = [
        base if base.name == chapter_folder else base / chapter_folder,
        base / "章节演示" / chapter_folder,
        base / "PySide6的学习版Web" / "章节演示" / chapter_folder,
    ]
    chapter_dir = next(
        (candidate for candidate in candidates if (candidate / "lesson.ipynb").exists()),
        None,
    )
    if chapter_dir is not None:
        break
if chapter_dir is None:
    raise FileNotFoundError(f"找不到章节目录：{chapter_folder}")

script = chapter_dir / relative_path
process = subprocess.Popen(
    [sys.executable, str(script), *[]],
    cwd=script.parent,
)
print('3.4 图形化应用程序示例', "已启动，进程号：", process.pid)

## 小练习

独立分析三个位置参数，不依赖前面的 notebook 状态。

In [ ]:
from PySide6.QtCore import QCommandLineParser


exercise_parser = QCommandLineParser()
exercise_parser.addPositionalArgument("action", "操作类型", "<action>")
exercise_parser.addPositionalArgument("input", "输入文件", "<input>")
exercise_parser.addPositionalArgument("output", "输出文件", "<output>")
exercise_parser.parse(["app.py", "copy", "input.txt", "backup.txt"])
action, input_path, output_path = exercise_parser.positionalArguments()
print(f"操作：{action}；从 {input_path} 到 {output_path}")

## 本章完成标准

能够运行需要的补充示例，并且在 `章节演示` 目录执行 `python3 check_notebook_coverage.py` 后本章显示 PASS。